In [10]:
import json
import os
import sys
from types import ModuleType

import numpy as np
import pandas as pd
import torch

from gluonts.dataset.pandas import PandasDataset
from gluonts.evaluation import make_evaluation_predictions
from lag_llama.gluon.estimator import LagLlamaEstimator


# ============================================================
# OPTIONAL COMPATIBILITY PATCH
# The uploaded Lag-Llama demo adds dummy classes for
# gluonts.torch.modules.loss to avoid import issues.
# Keep this block if your environment complains about that import.
# :contentReference[oaicite:1]{index=1}
# ============================================================
def create_dummy_module(module_path):
    parts = module_path.split(".")
    current = ""
    parent = None

    for part in parts:
        current = current + "." + part if current else part
        if current not in sys.modules:
            module = ModuleType(current)
            sys.modules[current] = module
            if parent:
                setattr(sys.modules[parent], part, module)
        parent = current

    return sys.modules[module_path]


try:
    import gluonts.torch.modules.loss  # noqa: F401
except Exception:
    gluonts_module = create_dummy_module("gluonts.torch.modules.loss")

    class DistributionLoss:
        def __init__(self, *args, **kwargs):
            pass

        def __call__(self, *args, **kwargs):
            return 0.0

        def __getattr__(self, name):
            return lambda *args, **kwargs: None

    class NegativeLogLikelihood:
        def __init__(self, *args, **kwargs):
            pass

        def __call__(self, *args, **kwargs):
            return 0.0

        def __getattr__(self, name):
            return lambda *args, **kwargs: None

    gluonts_module.DistributionLoss = DistributionLoss
    gluonts_module.NegativeLogLikelihood = NegativeLogLikelihood


# ============================================================
# METRIC
# ============================================================
def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


# ============================================================
# LAG-LLAMA HELPER
# Based on the uploaded demo:
# - load ckpt
# - read ckpt["hyper_parameters"]["model_kwargs"]
# - build estimator from checkpoint args
# - create module/transformation/predictor
# - call make_evaluation_predictions
# 
# ============================================================

def get_lag_llama_predictor(
    ckpt_path,
    prediction_length,
    context_length,
    num_samples=20,
    batch_size=64,
    device=None,
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    estimator_args = ckpt["hyper_parameters"]["model_kwargs"]

    ckpt_lags_seq = estimator_args.get("lags_seq", None)

    print("checkpoint input_size:", estimator_args.get("input_size"))
    print("checkpoint lags_seq type:", type(ckpt_lags_seq))
    if ckpt_lags_seq is not None:
        print("first 10 checkpoint lags:", ckpt_lags_seq[:10])

    # IMPORTANT:
    # use a dummy safe freq string only to survive estimator __init__
    estimator = LagLlamaEstimator(
        ckpt_path=ckpt_path,
        prediction_length=prediction_length,
        context_length=context_length,

        input_size=estimator_args["input_size"],
        n_layer=estimator_args["n_layer"],
        n_embd_per_head=estimator_args["n_embd_per_head"],
        n_head=estimator_args["n_head"],
        scaling=estimator_args["scaling"],
        time_feat=estimator_args["time_feat"],

        # dummy just for constructor
        lags_seq=["D"],

        nonnegative_pred_samples=True,
        rope_scaling={
            "type": "linear",
            "factor": max(
                1.0,
                (context_length + prediction_length) / estimator_args["context_length"]
            ),
        },
        batch_size=batch_size,
        num_parallel_samples=num_samples,
        device=device,
    )

    # overwrite with checkpoint lag indices AFTER init
    if ckpt_lags_seq is not None:
        estimator.lags_seq = ckpt_lags_seq

    lightning_module = estimator.create_lightning_module()
    transformation = estimator.create_transformation()
    predictor = estimator.create_predictor(transformation, lightning_module)

    return predictor

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"
CKPT_PATH = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\lag-llama\lag-llama.ckpt"

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
]

prediction_length = 96     # 24h ahead for 15-min data
context_length = 512       # good starting point; can test 32, 64, 128, 256, 512, 1024
num_samples = 20
batch_size = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

predictor = get_lag_llama_predictor(
    ckpt_path=CKPT_PATH,
    prediction_length=prediction_length,
    context_length=context_length,
    num_samples=num_samples,
    batch_size=batch_size,
    device=device,
)

rmse_results = []

for country in countries:
    print(f"\nProcessing country: {country}")

    data_path = os.path.join(DATA_DIR, f"dataset_{country}.csv")
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    # only households, exclude weather features
    households = [col for col in df.columns if col not in features]

    for day in days:
        print(f"   Day: {day}")

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            # training slice only before cutoff
            s_train = (
                df.loc[df.index < cutoff, household]
                .dropna()
                .sort_index()
                .astype(np.float32)
            )

            # skip too-short series
            if len(s_train) < max(context_length, 32):
                print(f"      Skipping {household}: too short ({len(s_train)} rows)")
                continue

            # build one-series GluonTS dataset
            long_df = pd.DataFrame({
                "item_id": household,
                "timestamp": s_train.index,
                "target": s_train.to_numpy(dtype=np.float32),
            })

            dataset = PandasDataset.from_long_dataframe(
                long_df,
                item_id="item_id",
                timestamp="timestamp",
                target="target",
                freq="15min",   # your data appears to be 15-minute resolution
            )

            forecast_it, ts_it = make_evaluation_predictions(
                dataset=dataset,
                predictor=predictor,
                num_samples=num_samples,
            )

            forecasts = list(forecast_it)
            if len(forecasts) == 0:
                print(f"      No forecast returned for {household}")
                continue

            forecast = forecasts[0]

            # Use mean forecast as point prediction
            # forecast.samples shape: (num_samples, prediction_length)
            y_pred = forecast.samples.mean(axis=0)

            # forecast start timestamp
            pred_index = pd.date_range(
                start=pd.Timestamp(forecast.start_date.to_timestamp()),
                periods=prediction_length,
                freq="15min",
            )

            # initialize output df once
            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=pred_index)

            predictions_df_all_households[household] = y_pred

            # align truth
            y_true = df.reindex(pred_index)[household].to_numpy()

            # keep only non-missing aligned entries
            mask = ~np.isnan(y_true)
            if mask.sum() == 0:
                print(f"      No valid truth values for {household}")
                continue

            rmse = root_mean_squared_error(y_true[mask], y_pred[mask])
            rmse_households.append(rmse)

        # average RMSE across households
        avg_rmse_households = float(np.mean(rmse_households)) if len(rmse_households) else np.nan

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        # save predictions
        if predictions_df_all_households is not None:
            output = os.path.join(
                OUT_DIR,
                f"LagLlamaUnivar_pred_{day}_{country}.csv"
            )
            os.makedirs(os.path.dirname(output), exist_ok=True)
            predictions_df_all_households.to_csv(output, index=True)
            print("      Saved:", output)
        else:
            print("      No predictions were saved for this split.")

# summary
rmse_df = pd.DataFrame(rmse_results)

print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

Using device: cuda
checkpoint input_size: 1
checkpoint lags_seq type: <class 'list'>
first 10 checkpoint lags: [0, 7, 8, 10, 11, 12, 13, 14, 19, 20]

Processing country: Germany
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\LagLlamaUnivar_pred_day1_Germany.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\LagLlamaUnivar_pred_day2_Germany.csv
   Day: day3
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\LagLlamaUnivar_pred_day3_Germany.csv
   Day: day4
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\LagLlamaUnivar_pred_day4_Germany.csv
   Day: day5
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\LagLlamaUnivar_pred_day5_Germany.csv

Processing country: Ireland
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_